# 이커머스 데이터 분석 리포트 — 1. 전체 데이터 기본 탐색

이 노트북은 원본 CSV 전체를 읽고 **규모·자료형·결측치·중복·고객과 상품·날짜·주문 상태·이미지 연결**을 점검합니다.
각 단계는 목적 → 코드 → 실제 수치와 시사점 순서로 구성됩니다. 아직 결측치를 채우거나 행을 삭제하지 않습니다.

프로젝트의 `.venv`를 노트북 커널로 선택하고 위에서부터 모두 실행하세요. 프로젝트 루트 또는 이 노트북 폴더에서 실행할 수 있습니다.
분석에는 Pandas를 사용하며, IPython은 결과 표시용입니다. 날짜 해석은 확정 통계와 가정 통계를 구분합니다.

In [2]:
from pathlib import Path
import io
import pandas as pd
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset/raw/csv/ecommerce_retail_transactions_raw.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 루트 또는 하위 폴더에서 실행하세요.')
CSV_PATH = ROOT / 'dataset/raw/csv/ecommerce_retail_transactions_raw.csv'
MAPPING_PATH = ROOT / 'dataset/raw/images/image_mapping.csv'
# ID는 문자열로 보존하고, 모든 원본 값을 문자열로 읽어 형식 오류도 살펴봅니다.
raw = pd.read_csv(CSV_PATH, dtype='string')
if raw.empty:
    raise ValueError('분석할 거래 행이 없습니다.')
# 공백만 있는 값도 결측치로 보는 탐색용 복사본입니다. 원본 raw는 유지합니다.
df = raw.apply(lambda col: col.str.strip()).replace('', pd.NA)
def explain(text):
    display(Markdown('**결과 해석:** ' + text))
print('Pandas:', pd.__version__)
print('데이터:', CSV_PATH.relative_to(ROOT))

Pandas: 3.1.1
데이터: dataset/raw/csv/ecommerce_retail_transactions_raw.csv


## 1. 데이터 규모와 첫 행 확인
`shape`는 행·컬럼 수, `head()`는 첫 5개 거래를 보여줍니다. 여기서 전체 행 수는 고객 수나 고유 주문 수와 다릅니다.
자료형은 원본 형식 점검을 위해 우선 모두 문자열로 보존했습니다.

In [3]:
n_rows, n_cols = df.shape
display(pd.DataFrame({'항목': ['데이터 행 수', '컬럼 수'], '값': [n_rows, n_cols]}))
display(raw.head())
explain(f'전체 {n_rows:,}행, {n_cols}개 컬럼입니다. '
        f'미션의 규모 조건(1,000행 이상·8컬럼 이상)은 {"충족" if n_rows >= 1000 and n_cols >= 8 else "미충족"}합니다. '
        '다만 규모 충족만으로 고객 식별자와 날짜의 품질까지 보장되지는 않으므로 아래 검사가 필요합니다.')

,항목,값
0,데이터 행 수,12180
1,컬럼 수,13


,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Shipping_City,Country,Order_Status,Customer_Rating
0,ORD011416,CUST00003,2026-04-03,Books,Self-Help Book,3,9.16,5.0,Net Banking,London,UK,Delivered,5.0
1,ORD000974,CUST00269,10-15-2024,Books,Fiction Novel,1,34.24,5.0,NetBanking,Toronto,CA,Shipped,<NA>
2,ORD000617,CUST02075,17 Jun 2024,Home & Kitchen,Vacuum Cleaner,2,111.95,15.0,DEBIT CARD,Brisbane,Australia,Shipped,<NA>
3,ORD006437,CUST02109,16/10/2025,Beauty,Hair Dryer,1,31.84,0.0,Cash on Delivery,Birmingham,U.K.,Delivered,3.0
4,ORD000740,CUST00349,06 Apr 2026,Home & Kitchen,Non-Stick Pan,2,72.0,<NA>,Credit_Card,Montreal,CA,Delivered,<NA>


**결과 해석:** 전체 12,180행, 13개 컬럼입니다. 미션의 규모 조건(1,000행 이상·8컬럼 이상)은 충족합니다. 다만 규모 충족만으로 고객 식별자와 날짜의 품질까지 보장되지는 않으므로 아래 검사가 필요합니다.

## 2. 자료형과 기초 통계
`info()`로 컬럼별 값의 존재 여부를 확인합니다. 이어서 수량·단가·할인율·평점을 **탐색용으로만 숫자로 변환**한 뒤 `describe()`로 분포를 봅니다.
변환 실패는 결측으로 표시하고 별도로 집계합니다. `std`는 표본 표준편차, 25%·50%·75%는 사분위수이며 50%는 중앙값입니다.

In [4]:
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())
numeric_cols = ['Quantity', 'Unit_Price_USD', 'Discount_Percent', 'Customer_Rating']
numeric = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
conversion_failures = (df[numeric_cols].notna() & numeric.isna()).sum()
display(conversion_failures.rename('숫자 변환 실패 수').to_frame())
display(numeric.describe())
p = numeric['Unit_Price_USD']
explain(f'단가의 평균은 {p.mean():,.2f} USD, 중앙값은 {p.median():,.2f} USD, '
        f'표준편차는 {p.std():,.2f} USD입니다. '
        '평균과 중앙값의 차이 및 큰 최댓값은 고가 상품이나 입력 오류의 영향을 살펴볼 단서이며, 이 통계만으로 이상치 삭제를 결정하지 않습니다.')

<class 'pandas.DataFrame'>
RangeIndex: 12180 entries, 0 to 12179
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Order_ID          12180 non-null  string
 1   Customer_ID       12180 non-null  string
 2   Order_Date        12180 non-null  string
 3   Product_Category  12180 non-null  string
 4   Product_Name      12180 non-null  string
 5   Quantity          12180 non-null  string
 6   Unit_Price_USD    12180 non-null  string
 7   Discount_Percent  11445 non-null  string
 8   Payment_Method    12180 non-null  string
 9   Shipping_City     11678 non-null  string
 10  Country           12180 non-null  string
 11  Order_Status      12180 non-null  string
 12  Customer_Rating   6030 non-null   string
dtypes: string(13)
memory usage: 1.2 MB



,숫자 변환 실패 수
Quantity,0
Unit_Price_USD,0
Discount_Percent,0
Customer_Rating,0


,Quantity,Unit_Price_USD,Discount_Percent,Customer_Rating
count,12180.0,12180.0,11445.0,6030.0
mean,2.019458,61.55069,7.299257,3.896186
std,1.234956,52.53839,7.120143,1.113482
min,-1.0,3.01,0.0,1.0
25%,1.0,22.02,0.0,3.0
50%,2.0,42.765,5.0,4.0
75%,3.0,87.7125,10.0,5.0
max,5.0,249.84,25.0,5.0


**결과 해석:** 단가의 평균은 61.55 USD, 중앙값은 42.77 USD, 표준편차는 52.54 USD입니다. 평균과 중앙값의 차이 및 큰 최댓값은 고가 상품이나 입력 오류의 영향을 살펴볼 단서이며, 이 통계만으로 이상치 삭제를 결정하지 않습니다.

## 3. 결측치 확인
결측 개수와 전체 행 대비 비율을 함께 봅니다. 원래 비어 있던 값과 앞뒤 공백을 정리한 뒤 발견한 결측을 구분합니다.
평점 누락은 미평가, 할인율 누락은 미기록일 수 있으므로 의미를 확인하기 전에는 0으로 채우지 않습니다.

In [5]:
missing = pd.DataFrame({
    '원본 결측 수': raw.isna().sum(),
    '공백 포함 결측 수': df.isna().sum(),
    '결측 비율(%)': df.isna().mean().mul(100).round(2),
}).sort_values('공백 포함 결측 수', ascending=False)
display(missing)
top = missing.iloc[0]
if top['공백 포함 결측 수'] == 0:
    explain('전체 컬럼의 결측치는 0건입니다. 다음으로 형식 오류와 중복을 확인합니다.')
else:
    explain(f'결측이 가장 많은 컬럼은 {missing.index[0]}이며 {int(top["공백 포함 결측 수"]):,}건 '
            f'({top["결측 비율(%)"]:.2f}%)입니다. '
            '컬럼의 의미와 주문 상태에 따른 누락 패턴을 확인한 뒤 대치 여부를 결정해야 합니다.')
display(pd.crosstab(df['Order_Status'].fillna('(상태 누락)'),
                    df['Customer_Rating'].isna(), colnames=['평점 결측 여부']))

,원본 결측 수,공백 포함 결측 수,결측 비율(%)
Customer_Rating,6150,6150,50.49
Discount_Percent,735,735,6.03
Shipping_City,502,502,4.12
Order_ID,0,0,0.00
Customer_ID,0,0,0.00
Order_Date,0,0,0.00
Product_Category,0,0,0.00
Product_Name,0,0,0.00
Quantity,0,0,0.00
Unit_Price_USD,0,0,0.00


**결과 해석:** 결측이 가장 많은 컬럼은 Customer_Rating이며 6,150건 (50.49%)입니다. 컬럼의 의미와 주문 상태에 따른 누락 패턴을 확인한 뒤 대치 여부를 결정해야 합니다.

평점 결측 여부,False,True
Order_Status,,
Cancelled,0,1207
Delivered,6030,643
Pending,0,1416
Returned,0,1036
Shipped,0,1848


## 4. 중복과 식별자 점검
전체 컬럼이 같은 중복 행과 주문 번호만 같은 행은 다릅니다. `duplicated()` 기본값은 첫 행을 제외한 추가 중복 수입니다.
주문 번호가 반복되어도 여러 상품을 포함하는 주문일 수 있으므로 바로 삭제하지 않습니다.

In [6]:
exact_duplicates = int(raw.duplicated().sum())
normalized_duplicates = int(df.duplicated().sum())
order_counts = df['Order_ID'].value_counts()
repeated_order_ids = order_counts[order_counts > 1]
deduplicated = df.drop_duplicates()
remaining_counts = deduplicated['Order_ID'].value_counts()
remaining_repeated_ids = int(remaining_counts.gt(1).sum())
conflicting_customers = int(df.groupby('Order_ID')['Customer_ID'].nunique().gt(1).sum())
display(pd.Series({
    '원본 완전 중복 추가 행': exact_duplicates,
    '공백 정리 후 완전 중복 추가 행': normalized_duplicates,
    '고유 주문 ID(결측 제외)': df['Order_ID'].nunique(),
    '2행 이상 등장한 주문 ID 수': len(repeated_order_ids),
    '주문 ID 결측 행': df['Order_ID'].isna().sum(),
    '완전 중복 제거 후에도 반복되는 주문 ID 수': remaining_repeated_ids,
    '고객이 둘 이상인 주문 ID 수': conflicting_customers,
    '고객 ID 결측 행': df['Customer_ID'].isna().sum(),
}, name='건수').to_frame())
example_ids = repeated_order_ids.index[:5]
display(df[df['Order_ID'].isin(example_ids)].sort_values('Order_ID'))
explain(f'원본 완전 중복은 첫 행을 제외하고 {exact_duplicates:,}행이며, '
        f'반복 등장한 주문 ID는 {len(repeated_order_ids):,}개입니다. '
        f'공백 정리 후 완전 중복을 제외하면 반복 주문 ID는 {remaining_repeated_ids:,}개, 고객 충돌 주문 ID는 {conflicting_customers:,}개입니다. '
        'RFM 구매 횟수는 행 수 대신 고유 주문 수를 기준으로 계산하고, 반복 ID의 원인을 먼저 확인해야 합니다.')

,건수
원본 완전 중복 추가 행,180
공백 정리 후 완전 중복 추가 행,180
고유 주문 ID(결측 제외),12000
2행 이상 등장한 주문 ID 수,180
주문 ID 결측 행,0
완전 중복 제거 후에도 반복되는 주문 ID 수,0
고객이 둘 이상인 주문 ID 수,0
고객 ID 결측 행,0


,Order_ID,Customer_ID,Order_Date,Product_Category,Product_Name,Quantity,Unit_Price_USD,Discount_Percent,Payment_Method,Shipping_City,Country,Order_Status,Customer_Rating
108,ORD000283,CUST02279,04 May 2026,Beauty,Face Moisturizer,1,9.4,<NA>,UPI,Manchester,uk,Delivered,4.0
881,ORD000283,CUST02279,04 May 2026,Beauty,Face Moisturizer,1,9.4,<NA>,UPI,Manchester,uk,Delivered,4.0
41,ORD003676,CUST00639,06 Jan 2025,Grocery,Almonds 500g,1,11.7,5.0,UPI,Abu Dhabi,U.A.E,Delivered,5.0
2040,ORD003676,CUST00639,06 Jan 2025,Grocery,Almonds 500g,1,11.7,5.0,UPI,Abu Dhabi,U.A.E,Delivered,5.0
106,ORD005308,CUST02026,26/08/2024,Grocery,Basmati Rice 5kg,3,20.41,10.0,Credit_Card,Hyderabad,india,Delivered,3.0
987,ORD005308,CUST02026,26/08/2024,Grocery,Basmati Rice 5kg,3,20.41,10.0,Credit_Card,Hyderabad,india,Delivered,3.0
79,ORD005394,CUST00342,12 Nov 2024,Grocery,Green Tea Pack,2,4.34,10.0,DEBIT CARD,Montreal,canada,Delivered,4.0
8797,ORD005394,CUST00342,12 Nov 2024,Grocery,Green Tea Pack,2,4.34,10.0,DEBIT CARD,Montreal,canada,Delivered,4.0
50,ORD007171,CUST00040,01-14-2025,Electronics,Webcam,3,189.65,0.0,credit card,Toronto,CA,Delivered,4.0
6910,ORD007171,CUST00040,01-14-2025,Electronics,Webcam,3,189.65,0.0,credit card,Toronto,CA,Delivered,4.0


**결과 해석:** 원본 완전 중복은 첫 행을 제외하고 180행이며, 반복 등장한 주문 ID는 180개입니다. 공백 정리 후 완전 중복을 제외하면 반복 주문 ID는 0개, 고객 충돌 주문 ID는 0개입니다. RFM 구매 횟수는 행 수 대신 고유 주문 수를 기준으로 계산하고, 반복 ID의 원인을 먼저 확인해야 합니다.

## 5. 고객·상품·카테고리와 표기 차이
고객별 고유 주문 수는 향후 RFM의 Frequency를 계산할 수 있는지 확인하는 기초 지표입니다. 아직 취소·반품을 제외하지 않은 원본 기준입니다.
상품은 모델 ID가 없으므로 우선 `Product_Name`을 연결 키로 사용합니다. 같은 이름이 실제 동일 모델임을 보장하지는 않습니다.

In [7]:
customer_orders = df.dropna(subset=['Customer_ID', 'Order_ID']).groupby('Customer_ID')['Order_ID'].nunique()
repeat_customers = int(customer_orders.ge(2).sum())
display(pd.Series({
    '고유 고객 수': df['Customer_ID'].nunique(),
    '고유 상품명 수': df['Product_Name'].nunique(),
    '고유 카테고리 수': df['Product_Category'].nunique(),
    '고유 주문이 2개 이상인 고객 수': repeat_customers,
}, name='개수').to_frame())
display(df.groupby('Product_Category', dropna=False).agg(
    거래_행수=('Order_ID', 'size'), 상품명_수=('Product_Name', 'nunique')))
display(df.groupby(['Product_Category', 'Product_Name'], dropna=False).size().rename('거래 행 수').reset_index())
for col in ['Country', 'Payment_Method', 'Order_Status']:
    print(f'\n{col} 원본 표기별 빈도')
    display(df[col].value_counts(dropna=False).rename('행 수').to_frame())
explain(f'고유 고객 {df["Customer_ID"].nunique():,}명 중 고유 주문이 2개 이상인 고객은 '
        f'{repeat_customers:,}명입니다. 취소·반품 포함 수치이므로 실제 반복 구매 고객 수는 분석 대상 상태를 정한 뒤 다시 계산해야 합니다.')

,개수
고유 고객 수,3374
고유 상품명 수,46
고유 카테고리 수,8
고유 주문이 2개 이상인 고객 수,2985


,거래_행수,상품명_수
Product_Category,,
Beauty,1510,6
Books,1552,4
Electronics,1507,8
Fashion,1539,8
Grocery,1491,4
Home & Kitchen,1568,7
Sports,1456,5
Toys,1557,4


,Product_Category,Product_Name,거래 행 수
0,Beauty,Face Moisturizer,227
1,Beauty,Hair Dryer,244
2,Beauty,Lipstick,258
3,Beauty,Perfume,281
4,Beauty,Shampoo,259
5,Beauty,Sunscreen SPF50,241
6,Books,Cookbook,396
7,Books,Fiction Novel,405
8,Books,Kids Storybook,381
9,Books,Self-Help Book,370



Country 원본 표기별 빈도


,행 수
Country,
UAE,609
australia,599
DE,596
Germany,592
IN,588
U.A.E,588
germany,587
Australia,586
india,584



Payment_Method 원본 표기별 빈도


,행 수
Payment_Method,
paypal,709
net banking,708
PayPal,695
UPI,692
Net Banking,690
DEBIT CARD,680
NetBanking,658
U.P.I,651
Pay Pal,649



Order_Status 원본 표기별 빈도


,행 수
Order_Status,
Delivered,6673
Shipped,1848
Pending,1416
Cancelled,1207
Returned,1036


**결과 해석:** 고유 고객 3,374명 중 고유 주문이 2개 이상인 고객은 2,985명입니다. 취소·반품 포함 수치이므로 실제 반복 구매 고객 수는 분석 대상 상태를 정한 뒤 다시 계산해야 합니다.

## 6. 날짜 형식과 거래 기간
혼합 형식에 일괄 자동 추론을 적용하면 월·일이 뒤바뀔 수 있습니다. 아래는 명시적인 형식 목록으로 파싱합니다.

- 연도가 앞인 날짜와 영문 월 날짜는 명시된 형식으로 변환합니다.
- 연도가 뒤인 숫자 날짜는 월/일과 일/월을 모두 시도합니다. 둘 다 유효하지만 서로 다르면 **모호한 날짜**로 표시합니다.
- 확정 기간에서는 모호한 날짜를 제외합니다. 참고용으로만 `MM-DD-YYYY`, `DD/MM/YYYY`라는 표기 가정을 적용한 기간도 보여줍니다. 출처 확인 전 이 가정을 확정하지 않습니다.
- 변환 실패 예시를 출력하며 원본 날짜를 덮어쓰지 않습니다.

구분자별로 일/월만 유효한 행과 월/일만 유효한 행을 따로 집계합니다. 가정과 불일치가 발견되면 참고용 기간도 재검토해야 합니다.

In [8]:
dates = df['Order_Date']
parsed = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
for fmt in ['%Y-%m-%d', '%Y/%m/%d', '%d %b %Y']:
    candidate = pd.to_datetime(dates, format=fmt, errors='coerce')
    parsed = parsed.fillna(candidate)
ambiguous = pd.Series(False, index=df.index)
assumed = parsed.copy()
format_checks = []
for sep, preferred in [('-', '%m-%d-%Y'), ('/', '%d/%m/%Y')]:
    mask = dates.str.fullmatch(r'\d{1,2}' + sep + r'\d{1,2}' + sep + r'\d{4}', na=False)
    day = pd.to_datetime(dates.where(mask), format=f'%d{sep}%m{sep}%Y', errors='coerce')
    month = pd.to_datetime(dates.where(mask), format=f'%m{sep}%d{sep}%Y', errors='coerce')
    conflict = mask & day.notna() & month.notna() & day.ne(month)
    preferred_candidate = pd.to_datetime(dates.where(mask), format=preferred, errors='coerce')
    format_checks.append({
        '구분자': sep, '가정 형식': preferred,
        '일/월만 유효': int((mask & day.notna() & month.isna()).sum()),
        '월/일만 유효': int((mask & month.notna() & day.isna()).sum()),
        '가정과 불일치(다른 해석만 유효)': int((mask & preferred_candidate.isna() & (day.notna() | month.notna())).sum()),
    })
    ambiguous |= conflict
    parsed = parsed.fillna(day.fillna(month).mask(conflict))
    assumed = assumed.fillna(pd.to_datetime(dates.where(mask), format=preferred, errors='coerce'))
# 한쪽 해석만 유효한 날짜는 참고용 시리즈에도 보존합니다.
assumed = assumed.fillna(parsed)
failed = dates.notna() & parsed.isna() & ~ambiguous
display(pd.DataFrame(format_checks))
display(pd.Series({
    '날짜 결측 행' : int(dates.isna().sum()),
    '모호한 날짜 행': int(ambiguous.sum()),
    '형식 오류 또는 변환 실패 행': int(failed.sum()),
    '모호하지 않은 변환 성공 행': int(parsed.notna().sum()),
}, name='행 수').rename(index={'날짜 결측 행':'날짜 결측 행'}).to_frame())
display(pd.DataFrame({'집계 기준': ['모호한 날짜 제외', '표기 가정 적용(미확정)'],
                      '최초 거래일': [parsed.min(), assumed.min()],
                      '최종 거래일': [parsed.max(), assumed.max()]}))
display(pd.DataFrame({'원본 날짜': dates[ambiguous | failed],
                      '판정': ambiguous[ambiguous | failed].map({True:'월·일 모호', False:'변환 실패'})}).head(15))
explain(f'모호한 날짜가 {int(ambiguous.sum()):,}행, 변환 실패가 {int(failed.sum()):,}행입니다. '
        f'모호하지 않은 {int(parsed.notna().sum()):,}행의 기간은 {parsed.min()}부터 {parsed.max()}까지입니다. '
        'RFM의 최근 구매일과 월별 매출은 날짜 해석에 민감하므로 출처에서 형식을 확인한 뒤 전체 기간과 기준일을 확정해야 합니다.')

,구분자,가정 형식,일/월만 유효,월/일만 유효,가정과 불일치(다른 해석만 유효)
0,-,%m-%d-%Y,0,1494,0
1,/,%d/%m/%Y,1464,0,0


,행 수
날짜 결측 행,0
모호한 날짜 행,1802
형식 오류 또는 변환 실패 행,0
모호하지 않은 변환 성공 행,10378


,집계 기준,최초 거래일,최종 거래일
0,모호한 날짜 제외,2024-01-01,2026-06-30
1,표기 가정 적용(미확정),2024-01-01,2026-06-30


,원본 날짜,판정
5,11-09-2024,월·일 모호
7,10/07/2024,월·일 모호
10,06-05-2024,월·일 모호
13,12-09-2024,월·일 모호
32,07/02/2024,월·일 모호
33,07-08-2025,월·일 모호
35,02/06/2025,월·일 모호
43,03/07/2025,월·일 모호
46,02-10-2024,월·일 모호
55,01-10-2025,월·일 모호


**결과 해석:** 모호한 날짜가 1,802행, 변환 실패가 0행입니다. 모호하지 않은 10,378행의 기간은 2024-01-01 00:00:00부터 2026-06-30 00:00:00까지입니다. RFM의 최근 구매일과 월별 매출은 날짜 해석에 민감하므로 출처에서 형식을 확인한 뒤 전체 기간과 기준일을 확정해야 합니다.

## 7. 주문 상태와 숫자 값의 유효성
취소·반품·배송 중·배송 완료가 섞여 있는지 확인합니다. 여기서는 매출을 확정하거나 상태별 행을 제거하지 않습니다.
수량·단가가 0 이하이거나 할인율·평점 범위를 벗어난 값은 오류 후보이며, 반품 표현 방식 등 출처 설명과 함께 판단합니다.

In [9]:
status = df['Order_Status'].value_counts(dropna=False).rename('행 수').to_frame()
status['비율(%)'] = (status['행 수'] / n_rows * 100).round(2)
display(status)
invalid_rules = {
    '수량 0 이하': numeric['Quantity'].le(0),
    '수량이 정수가 아님': numeric['Quantity'].notna() & numeric['Quantity'].mod(1).ne(0),
    '단가 0 이하': numeric['Unit_Price_USD'].le(0),
    '할인율 0~100 범위 밖': numeric['Discount_Percent'].lt(0) | numeric['Discount_Percent'].gt(100),
    '평점 1~5 범위 밖': numeric['Customer_Rating'].lt(1) | numeric['Customer_Rating'].gt(5),
}
display(pd.Series({k:int(v.sum()) for k,v in invalid_rules.items()}, name='후보 행 수').to_frame())
display(pd.crosstab(df['Order_Status'].fillna('(상태 누락)'),
                    invalid_rules['수량 0 이하'], colnames=['수량 0 이하 여부']))
explain(f'주문 상태는 결측을 제외하고 {df["Order_Status"].nunique()}종류입니다. '
        f'수량 0 이하 후보는 {int(invalid_rules["수량 0 이하"].sum()):,}행입니다. '
        '상태별 구매 인정 기준과 숫자 오류 처리 기준을 먼저 정해야 실제 매출과 RFM을 과대 계산하지 않습니다.')

,행 수,비율(%)
Order_Status,,
Delivered,6673,54.79
Shipped,1848,15.17
Pending,1416,11.63
Cancelled,1207,9.91
Returned,1036,8.51


,후보 행 수
수량 0 이하,192
수량이 정수가 아님,0
단가 0 이하,0
할인율 0~100 범위 밖,0
평점 1~5 범위 밖,0


수량 0 이하 여부,False,True
Order_Status,,
Cancelled,1189,18
Delivered,6563,110
Pending,1390,26
Returned,1017,19
Shipped,1829,19


**결과 해석:** 주문 상태는 결측을 제외하고 5종류입니다. 수량 0 이하 후보는 192행입니다. 상태별 구매 인정 기준과 숫자 오류 처리 기준을 먼저 정해야 실제 매출과 RFM을 과대 계산하지 않습니다.

## 8. 전체 상품과 이미지 연결 검사
이미지는 첫 100행에 나온 상품명을 기준으로 만든 합성 스케치입니다. 전체 CSV의 상품명과 비교하고 매핑 키 중복, 파일 부재, 경로 범위도 확인합니다.
같은 상품을 여러 번 거래해도 이미지 한 장을 공유합니다. 합성 스케치의 평균·표준편차는 실제 상품 외관이나 구매 원인을 나타내지 않습니다.

In [10]:
mapping = pd.read_csv(MAPPING_PATH, dtype='string')
mapping['image_path'] = mapping['image_path'].str.strip()
map_names = mapping['Product_Name'].str.strip()
if map_names.isna().any() or map_names.duplicated().any():
    raise ValueError('이미지 매핑의 상품명 누락 또는 중복을 먼저 해결하세요.')
def image_exists(value):
    if pd.isna(value):
        return False
    path = (ROOT / str(value)).resolve()
    return path.is_relative_to((ROOT / 'dataset/raw/images').resolve()) and path.is_file()
mapping['파일 존재'] = mapping['image_path'].map(image_exists)
mapping['Product_Name'] = map_names
available = set(mapping.loc[mapping['파일 존재'], 'Product_Name'])
products = set(df['Product_Name'].dropna())
missing_products = sorted(products - available)
extra_products = sorted(set(map_names.dropna()) - products)
unmatched = df['Product_Name'].isna() | ~df['Product_Name'].isin(available)
display(pd.Series({
    '전체 고유 상품명': len(products), '매핑 상품명': len(mapping),
    '파일이 없는 매핑': int((~mapping['파일 존재']).sum()),
    '이미지 미연결 고유 상품명': len(missing_products),
    '이미지 미연결 거래 행(상품명 결측 포함)': int(unmatched.sum()),
    'CSV에 없는 매핑 상품명': len(extra_products),
}, name='개수').to_frame())
display(pd.DataFrame({'이미지 추가 필요 상품명': missing_products}))
display(mapping.head())
explain(f'전체 상품명 {len(products)}개 중 이미지가 없는 상품명은 {len(missing_products)}개이며, '
        f'미연결 거래는 {int(unmatched.sum()):,}행({unmatched.mean()*100:.2f}%)입니다. '
        '추가 품목 또는 상품명 누락을 해결하고 나서 이미지 피처를 거래에 연결해야 합니다.')

,개수
전체 고유 상품명,46
매핑 상품명,41
파일이 없는 매핑,0
이미지 미연결 고유 상품명,5
이미지 미연결 거래 행(상품명 결측 포함),1003
CSV에 없는 매핑 상품명,0


,이미지 추가 필요 상품명
0,Denim Jacket
1,Hoodie
2,LED Desk Lamp
3,Power Bank
4,USB-C Cable


,Product_Name,image_path,image_source,파일 존재
0,Action Figure,dataset/raw/images/action_figure.png,AI-generated representative sketch,True
1,Air Fryer,dataset/raw/images/air_fryer.png,AI-generated representative sketch,True
2,Almonds 500g,dataset/raw/images/almonds_500g.png,AI-generated representative sketch,True
3,Backpack,dataset/raw/images/backpack.png,AI-generated representative sketch,True
4,Basmati Rice 5kg,dataset/raw/images/basmati_rice_5kg.png,AI-generated representative sketch,True


**결과 해석:** 전체 상품명 46개 중 이미지가 없는 상품명은 5개이며, 미연결 거래는 1,003행(8.23%)입니다. 추가 품목 또는 상품명 누락을 해결하고 나서 이미지 피처를 거래에 연결해야 합니다.

## 9. 다음 단계로 넘길 점검 요약
아래 표는 실행 결과로 갱신됩니다. 아직 값을 대치하거나 중복·이상치를 삭제하지 않았습니다.
이번 탐색에서 확인한 문제를 근거로 전처리 기준을 정하고, 이후 `DataAnalyzer` 클래스에 옮깁니다.

In [11]:
summary = pd.DataFrame([
    ['결측치', f'{int(df.isna().sum().sum()):,}개 셀', '그룹별 대치 대상과 미기록 유지 대상을 구분'],
    ['완전 중복', f'{exact_duplicates:,}개 추가 행', '생성 원인 확인 후 제거 여부 결정'],
    ['날짜 모호성', f'{int(ambiguous.sum()):,}행', '출처 형식 확인 후 파싱 규칙 확정'],
    ['날짜 변환 실패', f'{int(failed.sum()):,}행', '원본 표기 확인 및 실패 처리 기준 수립'],
    ['주문 상태', f'{df["Order_Status"].nunique()}종', 'RFM·매출에 포함할 상태 결정'],
    ['이미지 누락', f'{len(missing_products)}개 상품명 / {int(unmatched.sum()):,}행', '누락 품목·상품명 점검'],
], columns=['항목', '탐색 결과', '다음 작업'])
display(summary)
explain(f'총 {n_rows:,}행을 탐색했습니다. 다음 단계에서는 날짜·주문 상태·결측치 기준을 문서화하고 전처리를 구현합니다. '
        '이번 결과는 원본 기준이므로 정제 후 고객 수와 거래 기간을 다시 확인해야 합니다.')

,항목,탐색 결과,다음 작업
0,결측치,"7,387개 셀",그룹별 대치 대상과 미기록 유지 대상을 구분
1,완전 중복,180개 추가 행,생성 원인 확인 후 제거 여부 결정
2,날짜 모호성,"1,802행",출처 형식 확인 후 파싱 규칙 확정
3,날짜 변환 실패,0행,원본 표기 확인 및 실패 처리 기준 수립
4,주문 상태,5종,RFM·매출에 포함할 상태 결정
5,이미지 누락,"5개 상품명 / 1,003행",누락 품목·상품명 점검


**결과 해석:** 총 12,180행을 탐색했습니다. 다음 단계에서는 날짜·주문 상태·결측치 기준을 문서화하고 전처리를 구현합니다. 이번 결과는 원본 기준이므로 정제 후 고객 수와 거래 기간을 다시 확인해야 합니다.